# Thermal post-processing

Load the solved COMSOL model and export temperature and magnetic flux density fields for plotting.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import csv
import numpy as np
import mph

Start the COMSOL client and load the solved model

In [3]:
client = mph.start()

In [4]:
model_file = Path('E_quartercore_thermal_solved.mph').resolve()
if not model_file.is_file():
    raise FileNotFoundError(f'Run 1_thermal_simulation_run.ipynb first: {model_file}')
model = client.load(model_file)

Choose the COMSOL expressions and solution dataset

In [5]:
print('Available datasets:', model.datasets())
dataset_name = None  # Set a dataset name if the default is not the coupled thermal solution.
temperature_expression = 'T'
magnetic_flux_density_expression = 'mf.normB'

Available datasets: ['Grid 2D 1', 'Study (EM + Thermal)//Solution 1', 'Study (EM + Thermal)//Solution Store 1', 'Study (EM + Thermal)//Solution Store 2']


Evaluate coordinates, temperature in °C, and magnetic flux density in mT

In [6]:
if 'model' not in globals():
    raise RuntimeError('Run the notebook cells from the top first.')
expressions = ['x', 'y', 'z', temperature_expression, magnetic_flux_density_expression]
units = ['mm', 'mm', 'mm', 'degC', 'mT']
values = model.evaluate(expressions, unit=units, dataset=dataset_name, inner='last')
x_mm, y_mm, z_mm, temperature_c, magnetic_flux_density_mt = [np.asarray(value).ravel() for value in values]
lengths = {array.size for array in (x_mm, y_mm, z_mm, temperature_c, magnetic_flux_density_mt)}
if len(lengths) != 1:
    raise ValueError(f'COMSOL returned field arrays with different lengths: {sorted(lengths)}')
finite = np.isfinite(x_mm) & np.isfinite(y_mm) & np.isfinite(z_mm) & np.isfinite(temperature_c) & np.isfinite(magnetic_flux_density_mt)
x_mm, y_mm, z_mm, temperature_c, magnetic_flux_density_mt = [array[finite] for array in (x_mm, y_mm, z_mm, temperature_c, magnetic_flux_density_mt)]
summary = {
    'model': model_file.name,
    'dataset': dataset_name or 'default',
    'sample_count': int(temperature_c.size),
    'temperature_min_degC': float(temperature_c.min()),
    'temperature_max_degC': float(temperature_c.max()),
    'magnetic_flux_density_min_mT': float(magnetic_flux_density_mt.min()),
    'magnetic_flux_density_max_mT': float(magnetic_flux_density_mt.max()),
}
summary

{'model': 'E_quartercore_thermal_solved.mph',
 'dataset': 'default',
 'sample_count': 133278,
 'temperature_min_degC': 60.0,
 'temperature_max_degC': 155.29089429567733,
 'magnetic_flux_density_min_mT': 3.6774090597332144e-05,
 'magnetic_flux_density_max_mT': 616.1588594246122}

Save the summary and field data

In [7]:
global_dir = Path('global_data')
field_dir = Path('field_data')
global_dir.mkdir(exist_ok=True)
field_dir.mkdir(exist_ok=True)
summary_file = global_dir / 'post_processing_thermal.csv'
with summary_file.open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=list(summary))
    writer.writeheader()
    writer.writerow(summary)
field_file = field_dir / 'thermal_fields.npz'
np.savez_compressed(field_file, x_mm=x_mm, y_mm=y_mm, z_mm=z_mm, temperature_c=temperature_c, magnetic_flux_density_mt=magnetic_flux_density_mt)
print('Saved:', summary_file.resolve())
print('Saved:', field_file.resolve())

Saved: C:\Users\abujazar\github_repos\MPhSweepKit\examples\project_sst\thermal\quarter_block_winding\global_data\post_processing_thermal.csv
Saved: C:\Users\abujazar\github_repos\MPhSweepKit\examples\project_sst\thermal\quarter_block_winding\field_data\thermal_fields.npz


Release the COMSOL client

In [8]:
client.remove(model)
client.clear()
client.disconnect()